# Smart Budget — SageMaker Endpoint

**Ticket:** DATA-1179 · **Endpoint:** `smart-budget-suggestion-endpoint-{ENV}`

> **Kernel:** usar el mismo kernel donde funciona el proyecto SAFE.

| Step | Qué hace |
|------|----------|
| 0 | Configurar entorno (`dev` o `alpha`) |
| 1 | Setup — sesión y rol |
| 2 | Empaquetar y subir `model.tar.gz` (sin CSV — datos se consultan de Athena en tiempo de inferencia) |
| 3 | Crear y desplegar endpoint |
| 4 | Probar endpoint |


---
## Step 0 — Elegir entorno

Cambiar `ENV` a `"dev"` o `"alpha"` antes de ejecutar el notebook.


---
## IAM Prerequisites

The SageMaker **execution role** must have the following permissions to allow the endpoint to query Athena at inference time:

| Permission | Purpose |
|---|---|
| `athena:StartQueryExecution` | Start Athena queries |
| `athena:GetQueryResults` | Retrieve query results |
| `athena:GetQueryExecution` | Check query status |
| `glue:GetTable` | Read Glue table schema |
| `glue:GetDatabase` | Read Glue database schema |
| `s3:GetObject`, `s3:PutObject`, `s3:ListBucket` | Access staging bucket and Glue data |

If these permissions are missing, the endpoint will return `ModelError` on the first invocation.


In [ ]:
# ╔══════════════════════════════════════════╗
# ║  CAMBIAR AQUÍ: "dev" | "alpha"           ║
# ╚══════════════════════════════════════════╝
ENV = "dev"

assert ENV in ("dev", "alpha"), f"ENV debe ser 'dev' o 'alpha', recibido: {ENV!r}"
print(f"Entorno seleccionado: {ENV}")


---
## Step 1 — Setup — sesión y rol


In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role, Session
import sagemaker
import boto3, json, os, shutil, tarfile, time
from pathlib import Path

sagemaker_session = Session()
role = get_execution_role()

S3_BUCKET     = 'blossom-analytics-safe-dev-nv'
S3_KEY        = f'smart_budget/endpoint/v1/{ENV}/model.tar.gz'
S3_URI        = f's3://{S3_BUCKET}/{S3_KEY}'
ENDPOINT_NAME = f'smart-budget-suggestion-endpoint-{ENV}'
ATHENA_S3_STAGING_DIR = 's3://blossom-analytics-datalake-alpha/datalake/gold/athena-metadata/'
ATHENA_REGION_NAME    = 'us-east-2'
ATHENA_DATABASE       = 'dlh_gold_dough_dev'
ATHENA_TABLE          = 'smart_budget_transactions'

print(f"Role    : {role}")
print(f"Region  : {sagemaker_session.boto_region_name}")
print(f"S3 URI  : {S3_URI}")
print(f"Endpoint: {ENDPOINT_NAME}")


---
## Step 2 — Empaquetar `model.tar.gz` y subir a S3


In [ ]:
REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())

SRC_SMART_BUDGET = REPO_ROOT / 'src' / 'smart_budget'
ARTIFACTS_DIR    = REPO_ROOT / 'notebooks' / 'model_artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

staging = ARTIFACTS_DIR / 'staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

# Paquete smart_budget (inference.py lo importa desde aquí)
shutil.copytree(SRC_SMART_BUDGET, staging / 'smart_budget')
# NOTE: ningún CSV se empaca — los datos se leen de Athena en cada invocación.

# Crear tarball
tar_path = ARTIFACTS_DIR / 'model.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(staging, arcname='.')
print(f'✅ model.tar.gz creado ({tar_path.stat().st_size / 1024 / 1024:.1f} MB)')

# Subir a S3
s3 = boto3.client('s3')
s3.upload_file(str(tar_path), S3_BUCKET, S3_KEY)
print(f'✅ Subido a s3://{S3_BUCKET}/{S3_KEY}')


---
## Step 3 — Crear y desplegar endpoint


In [ ]:
sk_model = SKLearnModel(
    model_data=S3_URI,
    role=role,
    entry_point='inference.py',
    source_dir=str(REPO_ROOT / 'src' / 'sagemaker'),
    framework_version='1.2-1',
    sagemaker_session=sagemaker_session,
    env={
        'ATHENA_S3_STAGING_DIR': ATHENA_S3_STAGING_DIR,
        'ATHENA_REGION_NAME':    ATHENA_REGION_NAME,
        'ATHENA_DATABASE':       ATHENA_DATABASE,
        'ATHENA_TABLE':          ATHENA_TABLE,
    },
)


In [ ]:
predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=ENDPOINT_NAME,
)
print(f'✅ Endpoint desplegado: {ENDPOINT_NAME}')


---
## Step 4 — Probar el endpoint


In [ ]:
runtime = sagemaker_session.boto_session.client('sagemaker-runtime')

def invoke(payload):
    r = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Body=json.dumps(payload),
    )
    return json.loads(r['Body'].read())


### Test 1 — Happy path: miembro con historial → sugerencias en múltiples categorías


In [ ]:
# Miembros con >1 categoría sugerida:
# dev:   11393, 10859, 11066, 12277, ...
# alpha: 385462, 586384, 593079, 385543, ...
TEST_MEMBER = '11393' if ENV == 'dev' else '385462'
TEST_PERIOD = '2026-05'

result = invoke({'idmember': TEST_MEMBER, 'period_id': TEST_PERIOD})
print(json.dumps(result, indent=2))
assert result['total_suggested'] is not None
assert len(result['suggestions']) > 1
print(f"✅ {len(result['suggestions'])} categorías sugeridas — total: ${result['total_suggested']:,.2f}")


### Test 2 — Miembro no existe → error 400


In [ ]:
import botocore
try:
    invoke({'idmember': '0000000', 'period_id': TEST_PERIOD})
    print('❌ Test 2 FALLÓ — debería haber lanzado error')
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print('✅ Test 2 — miembro inexistente → error')


### Test 3 — Campo period_id faltante → error 400


In [ ]:
try:
    invoke({'idmember': TEST_MEMBER})
    print('❌ Test 3 FALLÓ')
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print('✅ Test 3 — period_id faltante → error')


---
## ⚠️ Borrar endpoint cuando termines

Genera costo por hora.


In [ ]:
client = boto3.client('sagemaker')
try:
    client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f'✅ Endpoint eliminado: {ENDPOINT_NAME}')
except client.exceptions.ResourceNotFound:
    print('Endpoint no encontrado — puede que ya esté eliminado.')

try:
    client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print(f'✅ EndpointConfig eliminado: {ENDPOINT_NAME}')
except client.exceptions.ResourceNotFound:
    pass
